# logit-thermo 探索：3 つの谷（液体 ⇄ 長い周期の結晶 ⇄ 短い周期の結晶）は 25 本のモデルでも同じか（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 10〜20 分。
- 探索（予想なし）。A2-2・A2-5・A2-6 の 25 本の生成で、窓ごとに L・A・C のラベルを付け、割合・移り変わり・段ごとの ΔH を出して
  `A2_explore/three_univ_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：結晶の始まりの探索のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_hidden.py`・`a2_explore_three.py`・`a2_explore_three_univ.py`（`ac7e071aed2722d2…`）は埋め込み。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 3 つの谷は 25 本のモデルでも同じか（CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 3 つの谷は 25 本のモデルでも同じか（CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore three states universality', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCON': '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9'})
SRCON = _s['SRCON']
SRCHD = '"""探索：結晶の谷の中に、周期のない結晶（中間の状態）の谷が隠れていないか（2026-10-30 17:4x、CPU、判定しない）。\nscripts/a2_explore_onset.py（copy_marks）を実行したのと同じ名前空間で実行する。\n\n窓（WIN ＝ 64 歩、16 歩おき）ごとに：\n- r ＝ 写しの印（4 トークン）の割合（0〜1）\n- P ＝ 周期の強さ ＝ 2〜32 歩のずれ p で、窓の中で「p 歩前と同じトークン」の割合 m(p) の最大（探索 8.13 と同じ m(p)）\nラベル：液体 L（r < 0.25）、周期のない結晶 A（r ≥ 0.5 かつ P < 0.5）、周期のある結晶 C（r ≥ 0.5 かつ P ≥ 0.5）、そのあいだは「ー」。\n(r, P) の 2 次元の分布と、同じ本の中で隣り合う窓のラベルの移り変わり（L→A、L→C、A→C …）を数える。\nオストワルトの段階則の形なら、C へは L から直接ではなく A を通って入ることが多く、A は C より先に出る。\n"""\nimport numpy as np\n\nHD_WIN, HD_STEP, HD_PMAX = 64, 16, 32\n\n\ndef window_feats(seq):\n    c = copy_marks(seq).astype(float)\n    s = np.asarray(seq)\n    L = len(s)\n    out = []\n    for a in range(0, L - HD_WIN + 1, HD_STEP):\n        w = s[a:a + HD_WIN]\n        r = c[a:a + HD_WIN].mean()\n        m = [np.mean(w[p:] == w[:-p]) for p in range(2, HD_PMAX + 1)]\n        out.append((r, max(m)))\n    return np.array(out)\n\n\ndef label(r, P):\n    if r < 0.25:\n        return \'L\'\n    if r >= 0.5:\n        return \'C\' if P >= 0.5 else \'A\'\n    return \'-\'\n\n\ndef hidden(toks_list):\n    """toks_list：(本数, L) のトークン列の集まり → 2 次元の分布・ラベルの割合・移り変わり。"""\n    feats, trans = [], {}\n    for seq in toks_list:\n        f = window_feats(seq)\n        feats.append(f)\n        labs = [label(r, P) for r, P in f]\n        # \'-\' を飛ばして、意味のあるラベルの並びで隣り合うものを数える\n        labs2 = [x for x in labs if x != \'-\']\n        for x, y in zip(labs2[:-1], labs2[1:]):\n            if x != y:\n                trans[x + y] = trans.get(x + y, 0) + 1\n    F = np.concatenate(feats)\n    H, _, _ = np.histogram2d(F[:, 0], F[:, 1], bins=[np.linspace(0, 1, 6), np.linspace(0, 1, 6)])\n    labs = [label(r, P) for r, P in F]\n    frac = {k: float(np.mean([x == k for x in labs])) for k in \'LAC-\'}\n    cry = F[F[:, 0] >= 0.5, 1]\n    Ph = np.histogram(cry, bins=np.linspace(0, 1, 11))[0]\n    return dict(hist2d=(H / H.sum()).round(4).tolist(), frac=frac, trans=trans, P_hist_in_crystal=(Ph / max(1, Ph.sum())).round(3).tolist())\n'
SRCTH = '"""探索：3 つの谷 L ⇄ A ⇄ C の段ごとの速さ・ファントホッフの ΔH・塩の効き（2026-10-30 17:5x、CPU、判定しない）。\nscripts/a2_explore_onset.py（copy_marks）と scripts/a2_explore_hidden.py（window_feats・label）を実行したのと同じ名前空間で実行する。\n窓（64 歩、16 歩おき）のラベルの列で、「ー」（どちらとも言えない）は直前のラベルのままとみなす（履歴つき）。\n速さ k_XY ＝ X→Y の回数 ÷ X にいた窓の数（1 窓 ＝ 16 歩）。段ごとの平衡定数 K_LA ＝ k_LA ÷ k_AL、K_AC ＝ k_AC ÷ k_CA。\nファントホッフ：ln K を 1/T に重みつきで当てはめた傾き（ナット、1 回の移り変わりあたり）。回数が両向きそれぞれ 8 回以上の温度だけ。\n"""\nimport numpy as np\n\nTH_MIN = 8\nTH_BOOT, TH_SEED = 300, 20261122\n\n\ndef seq_counts(seq):\n    """→ dict：各状態にいた窓の数、各移り変わりの回数。"""\n    f = window_feats(seq)\n    labs, cur = [], \'L\'\n    for r, P in f:\n        x = label(r, P)\n        if x != \'-\':\n            cur = x\n        labs.append(cur)\n    occ = {s: 0 for s in \'LAC\'}; tr = {}\n    for x, y in zip(labs[:-1], labs[1:]):\n        occ[x] += 1\n        if x != y:\n            tr[x + y] = tr.get(x + y, 0) + 1\n    return occ, tr\n\n\ndef counts_by_doc(toks):\n    """toks：(nT, S, nD, L) → (nT, nD, 3 + 6)：[L, A, C にいた窓の数, LA, AL, AC, CA, LC, CL の回数]。"""\n    nT, S, nD, _ = toks.shape\n    keys = [\'LA\', \'AL\', \'AC\', \'CA\', \'LC\', \'CL\']\n    out = np.zeros((nT, nD, 9))\n    for k in range(nT):\n        for j in range(nD):\n            for s in range(S):\n                occ, tr = seq_counts(toks[k, s, j])\n                out[k, j, :3] += [occ[\'L\'], occ[\'A\'], occ[\'C\']]\n                out[k, j, 3:] += [tr.get(x, 0) for x in keys]\n    return out\n\n\ndef rates(tot):\n    """tot：(nT, 9) → 速さ（1 窓あたり）。"""\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        return dict(LA=tot[:, 3] / tot[:, 0], AL=tot[:, 4] / tot[:, 1], AC=tot[:, 5] / tot[:, 1], CA=tot[:, 6] / tot[:, 2],\n                    LC=tot[:, 7] / tot[:, 0], CL=tot[:, 8] / tot[:, 2])\n\n\ndef vh_step(tot, temps, fwd, bwd, ifw, ibw):\n    k = rates(tot)\n    ok = (tot[:, ifw] >= TH_MIN) & (tot[:, ibw] >= TH_MIN)\n    idx = np.flatnonzero(ok)\n    if len(idx) < 3:\n        return dict(ok=False, idx=idx.tolist())\n    y = np.log(k[fwd][idx] / k[bwd][idx]); x = 1 / np.asarray(temps)[idx]\n    w = 1 / (1 / tot[idx, ifw] + 1 / tot[idx, ibw])\n    X = np.stack([np.ones_like(x), x], 1)\n    a, b = np.linalg.solve(X.T @ (w[:, None] * X), X.T @ (w * y))\n    return dict(ok=True, dH=float(b), dS=float(-a), Tm=float(-b / a) if (b > 0 and a < 0) else float(\'nan\'), idx=idx.tolist(), lnK=y.tolist())\n\n\ndef three_state(cnt, temps, seed=TH_SEED):\n    tot = cnt.sum(1)\n    res = dict(LA=vh_step(tot, temps, \'LA\', \'AL\', 3, 4), AC=vh_step(tot, temps, \'AC\', \'CA\', 5, 6),\n               rates={k: v.tolist() for k, v in rates(tot).items()}, counts=tot.tolist())\n    nD = cnt.shape[1]\n    rng = np.random.default_rng(seed)\n    bs = {\'LA\': [], \'AC\': []}\n    for _ in range(TH_BOOT):\n        w = np.bincount(rng.integers(0, nD, nD), minlength=nD).astype(float)\n        t = np.tensordot(cnt, w, axes=([1], [0]))\n        for st, (f, b, i, j) in {\'LA\': (\'LA\', \'AL\', 3, 4), \'AC\': (\'AC\', \'CA\', 5, 6)}.items():\n            if not res[st][\'ok\']:\n                continue\n            kk = rates(t)\n            idx = res[st][\'idx\']\n            with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n                y = np.log(kk[f][idx] / kk[b][idx])\n            if not np.all(np.isfinite(y)):\n                continue\n            x = 1 / np.asarray(temps)[idx]; ww = 1 / (1 / np.maximum(t[idx, i], 1) + 1 / np.maximum(t[idx, j], 1))\n            X = np.stack([np.ones_like(x), x], 1)\n            bs[st].append(float(np.linalg.solve(X.T @ (ww[:, None] * X), X.T @ (ww * y))[1]))\n    for st in bs:\n        if res[st][\'ok\'] and len(bs[st]) >= 20:\n            res[st][\'dH_ci\'] = [float(np.percentile(bs[st], 2.5)), float(np.percentile(bs[st], 97.5))]\n    return res\n\n\ndef salt_steps(cnt_none, cnt_arm, use_T):\n    """use_T：温度の番号。→ 4 つの速さそれぞれの log(偽薬 ÷ なし) を、温度で平均。"""\n    a, n = rates(cnt_arm.sum(1)), rates(cnt_none.sum(1))\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        return {k: float(np.nanmean(np.log(a[k][use_T] / n[k][use_T]))) for k in (\'LA\', \'AL\', \'AC\', \'CA\')}\n'
SRCTU = '"""探索：3 つの谷（液体 L ⇄ 長い周期の結晶 A ⇄ 短い周期の結晶 C）は、25 本のモデルでも同じか（2026-10-30 18:0x、CPU、判定しない）。\nscripts/a2_explore_onset.py・scripts/a2_explore_hidden.py・scripts/a2_explore_three.py を実行したのと同じ名前空間で実行する。\n材料：A2-2・A2-5・A2-6 の生成（探索 8.12 と同じ 25 本、温度 0.6〜1.5、サンプル 2 × 記事 40）。温度が 0.6 からしかないので、L⇄A の融点は\n0.6 以上の温度だけで出した「⁺」の数（探索 A2-11 の T_m⁺ と同じく、本当の値より高めに出る）。\n本ごとに：温度 0.6・0.7 での L・A・C の窓の割合、温度 0.6 の移り変わりの回数（L→A、L→C、A→C、C→A、A→L、C→L）、\nC から出るうちの C→L の割合、C に入るうちの L→C の割合、L⇄A と A⇄C の ΔH と融点（⁺）。\n"""\nimport numpy as np\n\nTU_RUNS = {\n    \'A2_2\': [\'pythia-1b\', \'qwen2.5-0.5b\', \'qwen2.5-0.5b-inst\', \'qwen2.5-1.5b\', \'olmo2-1b\', \'smollm2-1.7b\', \'smollm2-1.7b-inst\'],\n    \'A2_5\': [\'pythia-70m\', \'pythia-160m\', \'pythia-410m\', \'pythia-1b\', \'pythia-1.4b\', \'pythia-2.8b\',\n             \'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\'],\n    \'A2_6\': [\'qwen2.5-0.5b\', \'qwen2.5-1.5b\', \'qwen2.5-3b\', \'smollm2-135m\', \'smollm2-360m\', \'smollm2-1.7b\',\n             \'qwen3-0.6b\', \'qwen3-1.7b\', \'qwen3-4b\'],\n}\nTU_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\n\n\ndef one_run(toks):\n    cnt = counts_by_doc(toks)                     # (15, nD, 9)\n    tot = cnt.sum(1)\n    keys = [\'LA\', \'AL\', \'AC\', \'CA\', \'LC\', \'CL\']\n    row = {}\n    for T in (0.6, 0.7):\n        k = TU_TEMPS.index(T)\n        occ = tot[k, :3] / tot[k, :3].sum()\n        row[f\'frac_{T}\'] = dict(L=float(occ[0]), A=float(occ[1]), C=float(occ[2]))\n    k6 = 0\n    tr = {x: int(tot[k6, 3 + i]) for i, x in enumerate(keys)}\n    row[\'trans_0.6\'] = tr\n    row[\'C_exit_to_L\'] = tr[\'CL\'] / (tr[\'CL\'] + tr[\'CA\']) if (tr[\'CL\'] + tr[\'CA\']) else None\n    row[\'C_entry_from_L\'] = tr[\'LC\'] / (tr[\'LC\'] + tr[\'AC\']) if (tr[\'LC\'] + tr[\'AC\']) else None\n    la = vh_step(tot, TU_TEMPS, \'LA\', \'AL\', 3, 4); ac = vh_step(tot, TU_TEMPS, \'AC\', \'CA\', 5, 6)\n    row[\'LA\'] = {k: la.get(k) for k in (\'ok\', \'dH\', \'Tm\')}; row[\'LA\'][\'temps\'] = [TU_TEMPS[i] for i in la[\'idx\']]\n    row[\'AC\'] = {k: ac.get(k) for k in (\'ok\', \'dH\', \'Tm\')}; row[\'AC\'][\'temps\'] = [TU_TEMPS[i] for i in ac[\'idx\']]\n    return row\n\n\ndef three_univ(base):\n    res = {}\n    for folder, names in TU_RUNS.items():\n        for n in names:\n            Z = np.load(f\'{base}/{folder}/{n}.npz\')\n            assert [float(t) for t in Z[\'temps\']] == TU_TEMPS\n            res[f\'{folder}/{n}\'] = one_run(Z[\'toks\'])\n    return res\n\n\ndef self_test_tu():\n    rng = np.random.default_rng(0)\n    toks = rng.integers(100, 30000, (15, 2, 4, 512))\n    # 1 本目の温度 0.6：途中から 7 の周期の短い結晶\n    toks[0, 0, 0, 200:] = np.tile([5, 6, 7, 8, 9, 10, 11], 60)[:312]\n    r = one_run(toks)\n    assert r[\'frac_0.6\'][\'C\'] > 0 and r[\'trans_0.6\'][\'CL\'] == 0\n    return \'OK\'\n'
assert hashlib.sha256(SRCHD.encode()).hexdigest() == 'a77e0c10cfd4eb2512660686d9648cfac42d9ac00a7226c8aca21ee8f550a2a6' and hashlib.sha256(SRCTH.encode()).hexdigest() == '7c7f4fa27b6a394efaacf10cc4b98c3e09bd6ca15bc62d42af70f7b1eca3a0ce' and hashlib.sha256(SRCTU.encode()).hexdigest() == 'ac7e071aed2722d22313632d3a8b555dbc257dbf4688e6663c049ca7dba878d7'
exec(SRCON)
exec(SRCHD)
exec(SRCTH)
exec(SRCTU)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('自己テスト：', self_test_onset(), self_test_tu())


In [ ]:
# セル3：数えて保存（結果は表示しない）
try:
    r = three_univ('/content/drive/MyDrive/logit-thermo')
    r = dict(runs=r, onset_sha256='0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', hidden_sha256='a77e0c10cfd4eb2512660686d9648cfac42d9ac00a7226c8aca21ee8f550a2a6', three_sha256='7c7f4fa27b6a394efaacf10cc4b98c3e09bd6ca15bc62d42af70f7b1eca3a0ce', three_univ_sha256='ac7e071aed2722d22313632d3a8b555dbc257dbf4688e6663c049ca7dba878d7')
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo' + '/A2_explore/three_univ_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 3 つの谷は 25 本のモデルでも同じかで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：3 つの谷の 25 本の要約を A2_explore/three_univ_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
